# L-shaped algorithm example

We illustrate in this notebook the example 2 in Chapter 5, page 188, from Birge and Louveaux (2011), "Introduction to Stochastic Programming", 2nd edition, Springer.

We use the library StochasticPrograms. Its original version is no longer maintained and no longer works with current versions of JuMP and MathOptInterface: in particular, its L-shaped method returned wrong solutions with status `OPTIMAL`. We therefore use a fork that fixes it, https://github.com/fbastin/StochasticPrograms.jl.

In [1]:
import Pkg

In [2]:
Pkg.activate(".")

  Activating project at `~/Git/Optimization/SP_Introduction/code/StochasticPrograms`


In [3]:
# the fixed fork, not the registered release: `Pkg.add("StochasticPrograms")` would switch back to the latter
Pkg.add(url = "https://github.com/fbastin/StochasticPrograms.jl", rev = "master")
Pkg.add("MathOptInterface")

    Updating git-repo `https://github.com/fbastin/StochasticPrograms.jl`
   Resolving package versions...
     Project No packages added to or removed from `~/Git/Optimization/SP_Introduction/code/StochasticPrograms/Project.toml`
    Manifest No packages added to or removed from `~/Git/Optimization/SP_Introduction/code/StochasticPrograms/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/Git/Optimization/SP_Introduction/code/StochasticPrograms/Project.toml`
    Manifest No packages added to or removed from `~/Git/Optimization/SP_Introduction/code/StochasticPrograms/Manifest.toml`


In [4]:
using MathOptInterface

In [5]:
Pkg.add("HiGHS")

   Resolving package versions...
     Project No packages added to or removed from `~/Git/Optimization/SP_Introduction/code/StochasticPrograms/Project.toml`
    Manifest No packages added to or removed from `~/Git/Optimization/SP_Introduction/code/StochasticPrograms/Manifest.toml`


In [6]:
using StochasticPrograms
using HiGHS

In [7]:
Pkg.status("HiGHS")

Status `~/Git/Optimization/SP_Introduction/code/StochasticPrograms/Project.toml`
  [87dc4568] HiGHS v1.26.0


First, we construct the 2-stage model.

In [8]:
@stochastic_model simple_model begin
    @stage 1 begin
        @decision(simple_model, x >= 0)
        @objective(simple_model, Min, 0*x)
    end
    @stage 2 begin
        @known(simple_model, x)
        @uncertain ξ
        @recourse(simple_model, 0 <= y[i in 1:2])
        @objective(simple_model, Min, y[1] + y[2])
        @constraint(simple_model, y[1] - y[2] == ξ - x)
    end
end

Two-Stage Stochastic Model

minimize f₀(x) + 𝔼[f(x,ξ)]
  x∈𝒳

where

f(x,ξ) = min  f(y; x, ξ)
              y ∈ 𝒴 (x, ξ)


We now define three scenarios of equal probability.

In [9]:
ξ1 = @scenario ξ = 1 probability = 1/3
ξ2 = @scenario ξ = 2 probability = 1/3
ξ3 = @scenario ξ = 4 probability = 1/3

ξ = [ξ1, ξ2, ξ3]

3-element Vector{Scenario{@NamedTuple{ξ::Int64}}}:
 Scenario with probability 0.3333333333333333
  ξ: 1
 Scenario with probability 0.3333333333333333
  ξ: 2
 Scenario with probability 0.3333333333333333
  ξ: 4

## Deterministic equivalent

In order to have a solvable program, we must instantiate the model. By default, `StochasticPrograms.jl` will generate the derministic equivalent form.

In [10]:
sp = instantiate(simple_model, ξ, optimizer = HiGHS.Optimizer)

Stochastic program with:
 * 1 decision variable
 * 2 recourse variables
 * 3 scenarios of type Scenario
Structure: Deterministic equivalent
Solver name: HiGHS

We can check the model by printing it on the screen.

In [11]:
println(sp)

Deterministic equivalent problem
Min 0.3333333333333333 y₁[1] + 0.3333333333333333 y₁[2] + 0.3333333333333333 y₂[1] + 0.3333333333333333 y₂[2] + 0.3333333333333333 y₃[1] + 0.3333333333333333 y₃[2]
Subject to
 x ∈ Decisions
 y₁[1] ∈ RecourseDecisions
 y₁[2] ∈ RecourseDecisions
 y₂[1] ∈ RecourseDecisions
 y₂[2] ∈ RecourseDecisions
 y₃[1] ∈ RecourseDecisions
 y₃[2] ∈ RecourseDecisions
 x ≥ 0
 y₁[1] ≥ 0
 y₁[2] ≥ 0
 y₂[1] ≥ 0
 y₂[2] ≥ 0
 y₃[1] ≥ 0
 y₃[2] ≥ 0
 x + y₁[1] - y₁[2] = 1
 x + y₂[1] - y₂[2] = 2
 x + y₃[1] - y₃[2] = 4
Solver name: HiGHS


We now solve it.

In [12]:
optimize!(sp)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
LP has 3 rows; 7 cols; 9 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [3e-01, 3e-01]
  Bound   [0e+00, 0e+00]
  RHS     [1e+00, 4e+00]
Solving LP with useful basis so presolve not used
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -1.0000008669e+00 Ph1: 3(3); Du: 1(1) 0.0s
          2     1.0000000000e+00 Pr: 0(0) 0.0s

Model status        : Optimal
Simplex   iterations: 2
Objective value     :  1.0000000000e+00
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


We can check the first stage solution with the method `value()`.

In [13]:
value(sp[1,:x])

2.0

## L-shaped

We now express the program as a 2-stage optimization problem, and explore various variants of the L-shaped decomposition algorithm.

### Multi cut

In [14]:
sp_lshaped = instantiate(simple_model, ξ, optimizer = LShaped.Optimizer)

Stochastic program with:
 * 1 decision variable
 * 2 recourse variables
 * 3 scenarios of type Scenario
Structure: Stage-decomposition
Solver name: L-shaped with disaggregate cuts

In [15]:
set_optimizer_attribute(sp_lshaped, MasterOptimizer(), HiGHS.Optimizer)
set_optimizer_attribute(sp_lshaped, SubProblemOptimizer(), HiGHS.Optimizer)

In [16]:
optimize!(sp_lshaped)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
LP has 2 rows; 3 cols; 4 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 1e+00]
  Bound   [0e+00, 0e+00]
  RHS     [2e-01, 1e+00]
Presolving model
0 rows, 0 cols, 0 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-2); columns 0(-3); nonzeros 0(-4) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Objective value     :  7.6664672433e-01
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00
Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
LP has 2 rows; 3 cols; 4 nonzeros
Coefficient ranges

L-Shaped Gap  Time: 0:00:02 (1 iterations)
           Objective: -Inf
   Early termination: DUAL_INFEASIBLE
      Number of cuts: 3
          Iterations: 1


The master problem is unbounded from below as $K_1$ is not bounded.

Let modify the first-stage feasible set by imposing a large bound on $x$, here $10^9$.

In [17]:
@stochastic_model simple_model begin
    @stage 1 begin
        @decision(simple_model, 0 <= x <= 1e9)
        @objective(simple_model, Min, 0*x)
    end
    @stage 2 begin
        @known(simple_model, x)
        @uncertain ξ
        @recourse(simple_model, 0 <= y[i in 1:2])
        @objective(simple_model, Min, y[1] + y[2])
        @constraint(simple_model, y[1] - y[2] == ξ - x)
    end
end

Two-Stage Stochastic Model

minimize f₀(x) + 𝔼[f(x,ξ)]
  x∈𝒳

where

f(x,ξ) = min  f(y; x, ξ)
              y ∈ 𝒴 (x, ξ)


In [18]:
sp_lshaped = instantiate(simple_model, ξ, optimizer = LShaped.Optimizer)
set_optimizer_attribute(sp_lshaped, MasterOptimizer(), HiGHS.Optimizer)
set_optimizer_attribute(sp_lshaped, SubProblemOptimizer(), HiGHS.Optimizer)

println(sp_lshaped)

First-stage 
Min 0
Subject to
 x ∈ Decisions
 x ≥ 0
 x ≤ 1000000000

Second-stage 
Subproblem 1 (p = 0.33):
Min y[1] + y[2]
Subject to
 x ∈ Known(value = 0.0)
 y[1] ∈ RecourseDecisions
 y[2] ∈ RecourseDecisions
 y[1] ≥ 0
 y[2] ≥ 0
 x + y[1] - y[2] = 1

Subproblem 2 (p = 0.33):
Min y[1] + y[2]
Subject to
 x ∈ Known(value = 0.0)
 y[1] ∈ RecourseDecisions
 y[2] ∈ RecourseDecisions
 y[1] ≥ 0
 y[2] ≥ 0
 x + y[1] - y[2] = 2

Subproblem 3 (p = 0.33):
Min y[1] + y[2]
Subject to
 x ∈ Known(value = 0.0)
 y[1] ∈ RecourseDecisions
 y[2] ∈ RecourseDecisions
 y[1] ≥ 0
 y[2] ≥ 0
 x + y[1] - y[2] = 4

Solver name: L-shaped with disaggregate cuts


We now solve the program.

In [19]:
optimize!(sp_lshaped)

┌ Warning: ProgressMeter by default refresh meters with additional information in IJulia via `IJulia.clear_output`, which clears all outputs in the cell. 
│  - To prevent this behaviour, do `ProgressMeter.ijulia_behavior(:append)`. 
│  - To disable this warning message, do `ProgressMeter.ijulia_behavior(:clear)`.
└ @ ProgressMeter ~/.julia/packages/ProgressMeter/N660J/src/ProgressMeter.jl:607
L-Shaped Gap  Time: 0:00:00 (3 iterations)
        Objective: 1.0000000099341073
              Gap: 0.0
   Number of cuts: 7
       Iterations: 3


The first-stage solution is

In [20]:
value(sp_lshaped[1,:x])

2.0000000298023224

### Single Cut

Let's check what happens when the single cut variant is used.

In [21]:
set_optimizer_attribute(sp_lshaped, Aggregator(), Aggregate())
optimize!(sp_lshaped)

┌ Warning: ProgressMeter by default refresh meters with additional information in IJulia via `IJulia.clear_output`, which clears all outputs in the cell. 
│  - To prevent this behaviour, do `ProgressMeter.ijulia_behavior(:append)`. 
│  - To disable this warning message, do `ProgressMeter.ijulia_behavior(:clear)`.
└ @ ProgressMeter ~/.julia/packages/ProgressMeter/N660J/src/ProgressMeter.jl:607
L-Shaped Gap  Time: 0:00:00 (5 iterations)
        Objective: 1.0
              Gap: 0.0
   Number of cuts: 4
       Iterations: 5


More iterations are needed, to obtain a similar first-stage solution.

In [22]:
value(sp_lshaped[1,:x])

1.9999999999999996